# FVG on EURUSD (H1)

Loads **EURUSD** hourly OHLC from `notebooks/data/EURUSD/H1/ohlcv.csv` (same `time,open,high,low,close,tick_volume,...` layout as other symbols under `notebooks/data`). That folder is typically gitignored — add the CSV locally or export from MT5 like `00_data_feching.ipynb`. Detects fair value gaps (same rules as `fvg_downloaded/FVG.ipynb`), then plots a few pilot windows. The last chart keeps **only large gaps** (90th percentile of gap height among FVGs in that slice).

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go


def resolve_eurusd_h1_csv() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "data" / "EURUSD" / "H1" / "ohlcv.csv",
        cwd / "notebooks" / "data" / "EURUSD" / "H1" / "ohlcv.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(
        "EURUSD H1 not found. Tried:\n" + "\n".join(str(p) for p in candidates)
    )


DATA_CSV = resolve_eurusd_h1_csv()
print("Using", DATA_CSV)

In [ ]:
def load_eurusd_h1(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    rename = {}
    for lo, hi in (("open", "Open"), ("high", "High"), ("low", "Low"), ("close", "Close")):
        if lo in df.columns and hi not in df.columns:
            rename[lo] = hi
    df = df.rename(columns=rename)
    vol_col = "tick_volume" if "tick_volume" in df.columns else "Volume"
    if vol_col in df.columns:
        df = df[df[vol_col] != 0]
    df = df.sort_values("time").reset_index(drop=True)
    return df


df = load_eurusd_h1(DATA_CSV)
df.head()

In [ ]:
def detect_fvg(data: pd.DataFrame, lookback_period: int = 10, body_multiplier: float = 1.5):
    """Return per-row FVG tag: None or ('bullish'|'bearish', gap_low, gap_high, index)."""
    fvg_list: list = [None, None]
    for i in range(2, len(data)):
        first_high = data["High"].iloc[i - 2]
        first_low = data["Low"].iloc[i - 2]
        middle_open = data["Open"].iloc[i - 1]
        middle_close = data["Close"].iloc[i - 1]
        third_low = data["Low"].iloc[i]
        third_high = data["High"].iloc[i]

        prev_bodies = (
            data["Close"].iloc[max(0, i - 1 - lookback_period) : i - 1]
            - data["Open"].iloc[max(0, i - 1 - lookback_period) : i - 1]
        ).abs()
        avg_body_size = float(prev_bodies.mean())
        avg_body_size = avg_body_size if avg_body_size > 0 else 0.001

        middle_body = abs(middle_close - middle_open)

        if third_low > first_high and middle_body > avg_body_size * body_multiplier:
            fvg_list.append(("bullish", first_high, third_low, i))
        elif third_high < first_low and middle_body > avg_body_size * body_multiplier:
            fvg_list.append(("bearish", first_low, third_high, i))
        else:
            fvg_list.append(None)
    return fvg_list


df["FVG"] = detect_fvg(df)
n_fvg = df["FVG"].apply(lambda x: isinstance(x, tuple)).sum()
print("rows", len(df), "FVG count", int(n_fvg))

In [ ]:
def fvg_gap_height(fvg) -> float | None:
    if not isinstance(fvg, tuple):
        return None
    _, y0, y1, _ = fvg
    return abs(float(y1) - float(y0))


def plot_fvg_window(
    data: pd.DataFrame,
    start: int,
    end: int,
    title: str,
    *,
    big_only: bool = False,
    gap_quantile: float = 0.90,
) -> go.Figure:
    dfw = data.iloc[start:end]
    threshold = None
    if big_only:
        heights = [fvg_gap_height(r) for _, r in dfw.iterrows()]
        heights = [h for h in heights if h is not None]
        threshold = float(np.quantile(heights, gap_quantile)) if heights else 0.0

    fig = go.Figure()
    fig.add_trace(
        go.Candlestick(
            x=dfw.index,
            open=dfw["Open"],
            high=dfw["High"],
            low=dfw["Low"],
            close=dfw["Close"],
            name="Candles",
        )
    )

    for _, row in dfw.iterrows():
        if not isinstance(row["FVG"], tuple):
            continue
        fvg_type, y0, y1, idx = row["FVG"]
        gh = fvg_gap_height(row["FVG"])
        if big_only and threshold is not None and gh is not None and gh < threshold:
            continue
        color = "rgba(0,255,0,0.35)" if fvg_type == "bullish" else "rgba(255,0,0,0.35)"
        fig.add_shape(
            type="rect",
            x0=idx - 2,
            x1=idx + 30,
            y0=y0,
            y1=y1,
            fillcolor=color,
            opacity=0.85,
            layer="below",
            line=dict(width=0),
        )

    sub = f" (gap height ≥ {threshold:.5f})" if big_only and threshold is not None else ""
    fig.update_layout(
        title=title + sub,
        width=1200,
        height=700,
        xaxis=dict(showgrid=False),
        yaxis=dict(showgrid=False),
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
    )
    return fig

## Pilot charts (same logic as `fvg_downloaded/FVG.ipynb`, different index windows)

In [5]:
plot_fvg_window(df, 50, 540, "EURUSD H1 — window A").show()

In [ ]:
plot_fvg_window(df, 2500, 3100, "EURUSD H1 — window B").show()

In [7]:
plot_fvg_window(df, 9000, 9600, "EURUSD H1 — window C").show()

## Large FVGs only (within the slice: keep gaps at or above the 90th percentile of gap heights in that slice)

In [8]:
plot_fvg_window(df, 50, 540, "EURUSD H1 — window A", big_only=True, gap_quantile=0.90).show()